In [17]:
import requests
import json
import math

API_KEY = "33dfc12d7b4989174ea89bfa71c23540"

#调用高德POI的搜索接口
def search_poi(keywords, city, page=1, offset=25):
    url = "https://restapi.amap.com/v3/place/text"
    params = {
        "key": API_KEY,
        "keywords": keywords,
        "city": city,
        "citylimit": "true",
        "output": "json",
        "offset": offset,
        "page": page,
        "extensions": "base"
    }
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    data = response.json()

    if data['status'] == '1':
        return data
    else:
        print(f"API错误：{data.get('info','未知错误')}")
        return None


In [18]:
#数据清洗
# ── GCJ-02（火星坐标）→ WGS-84 转换 ──────────────────────────────────────
# 高德 API 返回的坐标是 GCJ-02，不能直接当作 WGS-84 使用，
# 否则叠加到 OSM/标准底图上会有 100-500 米偏移。

def _transform_lat(lng, lat):
    ret = -100.0 + 2.0*lng + 3.0*lat + 0.2*lat*lat + 0.1*lng*lat + 0.2*math.sqrt(abs(lng))
    ret += (20.0*math.sin(6.0*lng*math.pi) + 20.0*math.sin(2.0*lng*math.pi)) * 2.0 / 3.0
    ret += (20.0*math.sin(lat*math.pi)     + 40.0*math.sin(lat/3.0*math.pi)) * 2.0 / 3.0
    ret += (160.0*math.sin(lat/12.0*math.pi) + 320*math.sin(lat*math.pi/30.0)) * 2.0 / 3.0
    return ret

def _transform_lng(lng, lat):
    ret = 300.0 + lng + 2.0*lat + 0.1*lng*lng + 0.1*lng*lat + 0.1*math.sqrt(abs(lng))
    ret += (20.0*math.sin(6.0*lng*math.pi) + 20.0*math.sin(2.0*lng*math.pi)) * 2.0 / 3.0
    ret += (20.0*math.sin(lng*math.pi)     + 40.0*math.sin(lng/3.0*math.pi)) * 2.0 / 3.0
    ret += (150.0*math.sin(lng/12.0*math.pi) + 300.0*math.sin(lng/30.0*math.pi)) * 2.0 / 3.0
    return ret

def gcj02_to_wgs84(lng, lat):
    """将 GCJ-02 坐标转换为 WGS-84 坐标"""
    a  = 6378245.0
    ee = 0.00669342162296594323
    # 若不在中国大陆范围内则直接返回（海外 POI 本身就是 WGS-84）
    if not (72.004 < lng < 137.8347 and 0.8293 < lat < 55.8271):
        return lng, lat
    dlat      = _transform_lat(lng - 105.0, lat - 35.0)
    dlng      = _transform_lng(lng - 105.0, lat - 35.0)
    rad_lat   = lat / 180.0 * math.pi
    magic     = 1 - ee * math.sin(rad_lat) ** 2
    sqrt_magic = math.sqrt(magic)
    dlat = (dlat * 180.0) / ((a * (1 - ee)) / (magic * sqrt_magic) * math.pi)
    dlng = (dlng * 180.0) / (a / sqrt_magic * math.cos(rad_lat) * math.pi)
    return lng - dlng, lat - dlat

def parse_poi(poi_item):
    """
    解析单条 POI 记录，提取关键字段。
    坐标从 GCJ-02 转换为 WGS-84。
    """
    location = poi_item.get('location', '0,0')
    gcj_lng, gcj_lat = map(float, location.split(','))
    # ⚠️ 高德返回 GCJ-02，必须转换为 WGS-84 才能与标准底图对齐
    wgs_lng, wgs_lat = gcj02_to_wgs84(gcj_lng, gcj_lat)

    return {
        'id':        poi_item.get('id', ''),
        'name':      poi_item.get('name', ''),
        'type':      poi_item.get('type', ''),
        'typecode':  poi_item.get('typecode', ''),
        'address':   poi_item.get('address', ''),
        'district':  poi_item.get('adname', ''),
        'tel':       poi_item.get('tel', ''),
        'longitude': wgs_lng,
        'latitude':  wgs_lat,
    }

# 解析第一页数据
if result:
    import pandas as pd
    pois_page1 = [parse_poi(p) for p in result['pois']]
    df_sample  = pd.DataFrame(pois_page1)
    print(df_sample[['name', 'district', 'longitude', 'latitude']].head())

                name district   longitude   latitude
0   海底捞火锅(扬名广场夜宵主题店)      香洲区  113.574646  22.278826
1      隆都四季香饭店(碧海路店)      香洲区  113.575665  22.275156
2  幸福湘遇•湘菜·海鲜(情侣中路店)      香洲区  113.574432  22.272536
3        小蘭居餐厅(顺德鱼生)      香洲区  113.574691  22.272821
4       七叔公饭庄(豪园大厦店)      香洲区  113.574667  22.274915


In [19]:
import time, math
import pandas as pd
from tqdm import tqdm
import os
import geopandas as gpd

def fetch_all_pois(keywords, city, max_records=500):
    """批量分页获取 POI 数据"""
    all_pois = []
    offset   = 25

    first_result = search_poi(keywords, city, page=1, offset=offset)
    if not first_result:
        return []

    total       = min(int(first_result['count']), max_records)
    total_pages = math.ceil(total / offset)
    print(f"【{keywords}】在{city}共 {first_result['count']} 条，将抓取前 {total} 条...")

    all_pois.extend([parse_poi(p) for p in first_result['pois']])

    # 翻页
    for page in tqdm(range(2, total_pages + 1), desc="抓取进度"):
        res = search_poi(keywords, city, page=page, offset=offset)
        if res and res['pois']:
            all_pois.extend([parse_poi(p) for p in res['pois']])
        else:
            print(f"第{page}页返回空，停止抓取")
            break
        time.sleep(0.3)
        if len(all_pois) >= max_records:
            break

    return all_pois[:max_records]

print("=== 开始分行政区抓取珠海美食数据 ===")
districts = ["香洲区", "斗门区", "金湾区"]
all_zhuhai_food = []

for district in districts:
    print(f"\n>>> 正在抓取：{district}")
    district_data = fetch_all_pois("中学", district, max_records=500)
    all_zhuhai_food.extend(district_data)

# 根据 POI ID 去重
seen_ids = set()
unique_zhuhai_food = []
for p in all_zhuhai_food:
    if p['id'] not in seen_ids:
        seen_ids.add(p['id'])
        unique_zhuhai_food.append(p)

print(f"\n✅ 抓取完毕！去重后最终获得有效数据: {len(unique_zhuhai_food)} 条")

# 转换为 GeoJSON 并保存
if unique_zhuhai_food:
    os.makedirs("output", exist_ok=True)
    geojson_path = "output/zhuhai_school.geojson"
    
    # 1. 转为 DataFrame
    df_food = pd.DataFrame(unique_zhuhai_food)
    
    # 2. 过滤掉无效坐标
    df_food = df_food[(df_food['longitude'] != 0) & (df_food['latitude'] != 0)]
    
    # 3. 构建 GeoDataFrame
    gdf_food = gpd.GeoDataFrame(
        df_food,
        geometry=gpd.points_from_xy(df_food['longitude'], df_food['latitude']),
        crs='EPSG:4326'  # 指定坐标系为 WGS84
    )
    
    # 4. 导出为 GeoJSON
    gdf_food.to_file(geojson_path, driver='GeoJSON')
    print(f"✅ 珠海美食数据已成功保存为 GeoJSON 格式至: {geojson_path}")
    
    # 打印各区统计
    print("\n--- 珠海各区美食数量统计 ---")
    print(df_food.groupby('district').size().sort_values(ascending=False).to_string())
else:
    print("未抓取到有效数据！")

=== 开始分行政区抓取珠海美食数据 ===

>>> 正在抓取：香洲区
【中学】在香洲区共 90 条，将抓取前 90 条...


抓取进度: 100%|██████████████████████████████████████████████████████████████████████████| 3/3 [00:01<00:00,  2.04it/s]



>>> 正在抓取：斗门区
【中学】在斗门区共 48 条，将抓取前 48 条...


抓取进度: 100%|██████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.16it/s]



>>> 正在抓取：金湾区
【中学】在金湾区共 27 条，将抓取前 27 条...


抓取进度: 100%|██████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.34it/s]


✅ 抓取完毕！去重后最终获得有效数据: 161 条
✅ 珠海美食数据已成功保存为 GeoJSON 格式至: output/zhuhai_school.geojson

--- 珠海各区美食数量统计 ---
district
香洲区    87
斗门区    47
金湾区    27
